# Sentinel v3 (8,000-scam stratified cap, clean text) — Colab run

Approved 2026-10-04. Track: REAL PUBLIC DATA. No simulator data, no placeholder tokens.

Runtime → Change runtime type → **T4 GPU** (L4/A100 also fine). Run the cells top to bottom with **Runtime → Run all**.

- Rebuilds the corpus from pinned datasets and **refuses to train** unless it is byte-identical to `sentinel/manifests/sentinel_real_v3_scam8k.build.json`, including the 8,000 sampled scam ids. Validation and every test file are byte-identical to v2, so the committed baseline applies unchanged.
- **Smoke test first:** 50 training messages, 5 optimiser steps, then scoring of 50 messages per test file. If `smoke_check.py` fails, the notebook **stops**; report the failure; do not change the experiment.
- Full run: hyperparameters fixed by the approved config (Qwen2.5-1.5B-Instruct, 4-bit LoRA r16/α32, 1 epoch, LR 2e-4, effective batch 16, max seq 384, seed 13). ≈ 600 optimiser steps; expect roughly 25–40 min on a T4 plus ≈ 25 min of scoring and JSON generation.
- The threshold is chosen by the pre-registered rule on the UCI calibration half; nothing in this notebook takes a threshold as input.
- At the end, download **`sentinel_real_v3_scam8k_results.zip`** (metrics, predictions, logs, manifests; no weights) and send it back. `sentinel_real_v3_scam8k_adapter.zip` holds the LoRA weights; keep it, don't commit it.

In [ ]:
import subprocess, sys
def sh(cmd):
    print('$', cmd, flush=True)
    r = subprocess.run(cmd, shell=True)
    if r.returncode != 0:
        raise SystemExit(f'STOPPED: command failed with exit code {r.returncode}: {cmd}')

In [ ]:
sh('nvidia-smi')
sh('git clone https://github.com/TanviSaraswat8/Chakravyuh.git')
import os; os.chdir('Chakravyuh')
sh('git log -1 --oneline')

In [ ]:
sh('pip -q install unsloth trl datasets')
sh('pip -q install -r backend/requirements.txt -r backend/requirements-data.txt')

## 1. Datasets (pinned commits, SHA-256 checked)

In [ ]:
sh('python scripts/fetch_real_datasets.py fetch imc25_smishing sp24_gateway_phishing india_spam_sms_junioralive')

## 2. Official UCI zip (upload the same `smsspamcollection.zip` as before)

In [ ]:
from google.colab import files
up = files.upload()
import hashlib, pathlib
z = pathlib.Path(next(iter(up)))
assert hashlib.sha256(z.read_bytes()).hexdigest() == '1587ea43e58e82b14ff1f5425c88e17f8496bfcdb67a583dbff9eefaf9963ce3', 'not the registered UCI zip'
sh(f'mkdir -p data/incoming && mv "{z}" data/incoming/smsspamcollection.zip')
sh('python scripts/register_official_uci.py data/incoming/smsspamcollection.zip')

## 3. Validate, splits, UCI calibration split, v3 build and the approval gate

In [ ]:
sh('python scripts/validate_real_dataset.py imc25_smishing sp24_gateway_phishing india_spam_sms_junioralive uci_sms_spam')
sh('python scripts/prepare_real_splits.py')
sh('python scripts/run_cross_source.py save-uci-split')
sh('python sentinel/build_sft_real.py --config sentinel/configs/sentinel_real_v3_scam8k.json')
sh('python sentinel/verify_build.py sentinel_real_v3_scam8k')   # must print BUILD MATCHES APPROVED MANIFEST

## 4. SMOKE TEST — 50 messages, 5 steps. The notebook stops here if anything fails.

In [ ]:
sh('python sentinel/finetune.py --base unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit --data sentinel/sft_real/sentinel_real_v3_scam8k --out sentinel/outputs/sentinel_real_v3_scam8k_smoke --max-seq 384 --epochs 1 --lr 2e-4 --rank 16 --batch 8 --grad-accum 2 --seed 13 --limit 50 --max-steps 5')
sh('python sentinel/predict_hf.py --model sentinel/outputs/sentinel_real_v3_scam8k_smoke/lora --data sentinel/sft_real/sentinel_real_v3_scam8k --out sentinel/outputs/sentinel_real_v3_scam8k_smoke/predictions --limit 50 --latency-n 20')
sh('python sentinel/smoke_check.py --run sentinel/outputs/sentinel_real_v3_scam8k_smoke --data sentinel/sft_real/sentinel_real_v3_scam8k --n 50')   # must print SMOKE TEST PASSED

## 5. Full training run (approved hyperparameters, unchanged)

In [ ]:
sh('mkdir -p sentinel/outputs/sentinel_real_v3_scam8k && nvidia-smi > sentinel/outputs/sentinel_real_v3_scam8k/hardware.txt && nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv >> sentinel/outputs/sentinel_real_v3_scam8k/hardware.txt')
sh('python sentinel/finetune.py --base unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit --data sentinel/sft_real/sentinel_real_v3_scam8k --out sentinel/outputs/sentinel_real_v3_scam8k --max-seq 384 --epochs 1 --lr 2e-4 --rank 16 --batch 8 --grad-accum 2 --seed 13')

## 6. Score every test file, evaluate per source, apply the pre-registered criteria

In [ ]:
sh('python sentinel/predict_hf.py --model sentinel/outputs/sentinel_real_v3_scam8k/lora --data sentinel/sft_real/sentinel_real_v3_scam8k --out sentinel/outputs/sentinel_real_v3_scam8k/predictions')
sh('python sentinel/evaluate_cross_source.py --data sentinel/sft_real/sentinel_real_v3_scam8k --pred sentinel/outputs/sentinel_real_v3_scam8k/predictions --out sentinel/outputs/sentinel_real_v3_scam8k/metrics_real.json')
# Exit code 1 here only means "criteria not met"; that is a result, not an error, so it does not stop the notebook.
subprocess.run('python sentinel/check_acceptance.py --sentinel sentinel/outputs/sentinel_real_v3_scam8k/metrics_real.json --baseline sentinel/baselines/sentinel_real_v3_scam8k/chakravyuh_tagger_arch/metrics.json --out sentinel/outputs/sentinel_real_v3_scam8k/acceptance.json', shell=True)

## 7. Manifest and downloads

In [ ]:
sh('python sentinel/write_model_manifest.py --run sentinel/outputs/sentinel_real_v3_scam8k --data sentinel/sft_real/sentinel_real_v3_scam8k --config sentinel/configs/sentinel_real_v3_scam8k.json')
sh('cp sentinel/sft_real/sentinel_real_v3_scam8k/build_manifest.json sentinel/outputs/sentinel_real_v3_scam8k/build_manifest.json && cp sentinel/sft_real/sentinel_real_v3_scam8k/sampled_scam_ids.json sentinel/outputs/sentinel_real_v3_scam8k/sampled_scam_ids.json')
sh('cp sentinel/outputs/sentinel_real_v3_scam8k_smoke/smoke_check.json sentinel/outputs/sentinel_real_v3_scam8k/smoke_check.json && cp sentinel/outputs/sentinel_real_v3_scam8k_smoke/training_log.json sentinel/outputs/sentinel_real_v3_scam8k/smoke_training_log.json')
sh('cd sentinel/outputs && zip -qr sentinel_real_v3_scam8k_results.zip sentinel_real_v3_scam8k -x "sentinel_real_v3_scam8k/lora/*" "sentinel_real_v3_scam8k/checkpoint-*" "sentinel_real_v3_scam8k/gguf/*"')
sh('cd sentinel/outputs && zip -qr sentinel_real_v3_scam8k_adapter.zip sentinel_real_v3_scam8k/lora')
from google.colab import files
files.download('sentinel/outputs/sentinel_real_v3_scam8k_results.zip')
files.download('sentinel/outputs/sentinel_real_v3_scam8k_adapter.zip')